# 記事作成ワークフローの実装

このノートブックでは、次の業務プロセスをADKのフレキシブルワークフローで実装します。

1. 企画担当者が記事のテーマを選定する。
2. 調査担当者がテーマに沿った情報を収集して調査レポートにまとめる。
3. 専門のライターが、調査レポートを参考にして原稿を書く。
4. 編集担当者が原稿をレビューして、改善点を指摘する。
5. 原稿の修正とレビューを繰り返して、企画担当者が満足する内容になったら完成とする。

調査担当者、ライター、編集者をエージェントに置き換えて自動化します。

## 事前準備

**[AWW-01]**

ADK と Gemini API の利用に必要なパッケージをインストールします。

ADK は開発の速度が速く、後方互換性のない機能変更が行われることがあります。ここでは、安全のために動作確認ができたバーションを指定してインストールしています。

**最後に `RESTART SESSION` ボタンが表示された場合は、これをクリックしてセッションを再起動してください。**

**注意**: `ERROR: pip's dependency resolver does not currently take into account...` のようなエラーメッセージが表示される場合がありますが、これは無視して構いません。

In [ ]:
%pip install --upgrade --user \
    google-adk==2.8.0 \
    google-cloud-aiplatform==2.0.1 \
    google-genai==2.20.0

**[AWW-02]**

インストールされたパッケージのバージョンを確認します。

In [1]:
!pip list | grep -E "(google-adk|google-genai|google-cloud-aiplatform)"

google-adk                            2.8.0
google-cloud-aiplatform               2.0.1
google-genai                          2.20.0


下記の内容が出力されたことを確認してください。

```
google-adk                               2.8.0
google-cloud-aiplatform                  2.0.1
google-genai                             2.20.0
```

## ユーザー認証

**[AWW-03]**

変数 `PROJECT_ID` に事前に準備したプロジェクトのプロジェクト ID を指定してください。

**注意**: プロジェクトを作成したユーザーアカウントと Colab を使用中のユーザーアカウントが一致している必要があります。


In [ ]:
###
PROJECT_ID = 'Project ID を入力'
###

if (not PROJECT_ID) or PROJECT_ID == 'Project ID を入力':
    print('Gemini API を使用する Google Cloud Project の Project ID を入力してください。')
else:
    print(f'変数 PROJECT_ID を設定しました。')
    print(f'PROJECT_ID = {PROJECT_ID}')

**[AWW-04]**

Google Cloud のプロジェクトを使用するためのユーザー認証を行います。

ポップアップ画面の指示に従って、認証処理を行ってください。

許可する操作を選択するチェックボックスが表示された場合は、すべてにチェックを入れてください。

In [3]:
from google.colab import auth
auth.authenticate_user(project_id=PROJECT_ID)

## 初期設定

**[AWW-05]**

この後の作業に必要なモジュールをインポートして、LlmAgent オブジェクトが参照する環境変数を設定します。

In [4]:
import os
from IPython.display import Markdown, display
from pydantic import BaseModel, Field
import agentplatform
from agentplatform.frameworks import AdkApp
from google.adk import Event, Workflow
from google.adk.agents.llm_agent import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.planners import BuiltInPlanner
from google.adk.tools import google_search
from google.adk.workflow import DEFAULT_ROUTE
from google.genai.types import (
    GenerateContentConfig, ThinkingConfig, ThinkingLevel,
    HttpOptions, HttpRetryOptions,
)

agentplatform.init(project=PROJECT_ID, location='us-central1')
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
os.environ['GOOGLE_CLOUD_LOCATION'] = 'global'
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'True'

**[AWW-06]**

AdkApp オブジェクトとの会話を行う簡易的なアプリケーションのクラス ChatClient を定義します。

ワークフローの進捗にあわせて結果を表示する `async_output` オプションを追加しています。

In [5]:
class ChatClient:
    def __init__(self, adk_app, user_id='default_user'):
        self.adk_app = adk_app
        self.user_id = user_id
        self.session_id = None

    async def async_stream_query(self, message, async_output=False):
        if not self.session_id:
            session = await self.adk_app.async_create_session(
                user_id=self.user_id,
            )
            self.session_id = session['id']

        result = []
        async for event in self.adk_app.async_stream_query(
            user_id=self.user_id,
            session_id=self.session_id,
            message=message,
        ):
            if ('content' in event and 'parts' in event['content']):
                response = '\n'.join(
                    [p['text'] for p in event['content']['parts'] if 'text' in p]
                )
                if response:
                    result.append(response)
                    if async_output:
                        display(Markdown(f'**[{event["author"]}]**'))
                        display(Markdown(response))
        if not async_output:
            return '\n'.join(result)

**[AWW-07]**

「日本語テキスト出力時のマークダウン表示崩れの防止ルール」と Gemini API のモデル選択をテンプレートとして用意しておきます。


In [6]:
markdown_rule = r'''
# 日本語テキスト出力時のマークダウン表示崩れの防止ルール
マークダウンの修飾記号（太字 `**` や斜体 `*` など）と、日本語の特殊記号や全角文字が隣り合うことで生じる表示崩れを防ぐため、以下のルールを厳守してください。

## 1. 記号と太字の間にスペースを入れる基本ルール
太字記号（`**`）の直前・直後に、鉤括弧や丸括弧、その他の記号が来る場合は、必ず半角スペースを1つ挟んでください。

- **全角鉤括弧（「」）：**
  - 【NG例】`**「重要」**なポイント`
  - 【OK例】`**「重要」** なポイント`
- **全角・半角の丸括弧（（） / ()）：**
  - 【NG例】`**（注意）**事項です`
  - 【OK例】`**（注意）** 事項です`
- **パーセント（% / ％）や数値・単位：**
  - 【NG例】`達成率は**100%**です`
  - 【OK例】`達成率は **100%** です`

## 2. 記号自体のエスケープと厳禁な組み合わせ
- マークダウンの特殊文字として誤認されやすい半角記号（`*`, `_`, `~`, `\` など）をそのまま表示する場合は、隣接するアスタリスク等との間にスペースを設けてください。
- アンダースコア（`_`）は単語の途中の日本語や全角文字と混ざると強力なイタリック・太字パーサに拾われやすいため、強調用以外では原則として直接隣接させないでください。
'''

flash_lite_model = Gemini(
    model='gemini-3.1-flash-lite',
    retry_options=HttpRetryOptions(),
)

flash_model = Gemini(
    model='gemini-3.8-flash',
    retry_options=HttpRetryOptions(),
)

planner = BuiltInPlanner(
    thinking_config=ThinkingConfig(
        thinking_level=ThinkingLevel.LOW,
        include_thoughts=False,
    ),
)

## タスクノードとエージェントノードの定義

**[AWW-08]**

記事のテーマとターゲット層を選定する託すノードを定義します。

In [7]:
class ArticleInformation(BaseModel):
    """記事の情報"""
    theme: str = Field(description='記事のテーマ')
    audience: str = Field(description='対象読者（ターゲット層）')

instruction = f'''
# タスク
1. ネットに掲載する記事の基本情報をユーザーにヒアリングします。
2. 得られた情報を ArticleInformation にセットします。
3. ArticleInformation の情報がそろったらタスクを終了します。

# 条件
- 必要に応じてユーザーの相談に乗って、google_search で参考情報を収集・提供してください。
- こちらから提案した内容は、ユーザーの確認を得た上で確定してください。

{markdown_rule}
'''

article_information_task = LlmAgent(
    name='article_information_task',
    model=flash_model,
    mode='task',
    output_schema=ArticleInformation,
    description='記事の基本情報を決めるタスク',
    instruction=instruction,
    tools=[google_search],
    planner=planner,
)

async def output_article_information(node_input: ArticleInformation):
    message = f'''
### 記事のテーマとターゲット層を決定しました。
- テーマ: {node_input.theme}
- ターゲット層: {node_input.audience}
'''
    return Event(
        node_output=node_input,
        message=message,
    )

**[AWW-09]**

調査レポートを作成するエージェントノードを定義します。

In [8]:
instruction = f'''
あなたの役割は、記事の執筆に必要な情報を収集して調査レポートにまとめる事です。
- 指定されたテーマとターゲット層の記事を執筆する際に参考となるトピックを5項目選定します。
- それぞれのトピックについて情報を深掘りして、トピックごとに5文以上の長さで調査結果をまとめます。
- トピックの選定と情報の深掘りでは、google_search で最新の情報を検索すること。

{markdown_rule}
'''

research_agent = LlmAgent(
    name='research_agent',
    model=flash_model,
    description='原稿執筆の参考となる調査レポートを作成するエージェント',
    instruction=instruction,
    input_schema=ArticleInformation,
    tools=[google_search],
)

**[AWW-10]**

記事の原稿を執筆するエージェントノードを定義します。

In [9]:
instruction = f'''
あなたの役割は、特定のテーマに関する気軽な読み物記事を書くことです。
記事の「テーマ」「ターゲット層」と、その内容に関連する「調査レポート」が与えられているので、調査レポートに記載の客観的事実に基づいて、信頼性のある記事を書いてください。

# 執筆方針
- トピックに関してある程度の基礎知識がある読者を前提として、数分で気軽に読める内容にしてください。
- 比較的カジュアルで語りかける口調の文章にします。
- 思考過程は出力せずに、最終結果だけを出力します。
- 記事タイトルは付けないで、次の構成で出力します。各セクションタイトルは、内容に合わせてアレンジしてください。
0. 導入：セクションタイトルを付けないで、この記事を読みたくなる導入を1〜2文にまとめます。
1. 概要：トピックの全体像をまとめて簡単に説明します。
2. 最新情報：特に注目したい新しい情報を取り上げます。
3. 実践：トピックに関して、読者自身がやってみるとよさそうな事を1つ紹介します。
4. まとめ

- 各セクションのタイトルはマークダウンヘッダ ## 付けます。必要に応じて箇条書きのマークダウンを使用します。
- それ以外のマークダウンによる装飾は行いません。

# レビュー結果の反映
- レビュアーによる修正ポイントが指摘された際は、直前に書いた記事を指摘に従って書き直します。
- 執筆方針に反する指示がある場合は、レビュアーの指示を優先します。
- これまでの原稿とレビュー結果の履歴がある場合は、その内容も考慮します。

{markdown_rule}
'''

writer_agent = LlmAgent(
    name='writer_agent',
    model=flash_model,
    description='記事の原稿を執筆するエージェント',
    instruction=instruction,
    include_contents='default',
    output_key='article_content',
)

**[AWW-11]**

記事の原稿をレビューするエージェントノードを定義します。

In [10]:
instruction = f'''
あなたの役割は、読み物記事をレビューして、記事の条件にあった内容にするための改善コメントを与える事です。

# 記事の条件
- 記事は、はじめに40文字程度のタイトルがあること。
　今日から役立つ生活情報があって「すぐに読まなきゃ」と読者が感じるタイトルにすること。
　タイトルはマークダウンヘッダ # をつけること。
- タイトルの直後に「なぜいまこのテーマを取り上げるのか」をまとめた導入を加えて、読者にこの記事を読む動機づけを与えます。
- 各セクションのサブタイトルには、絵文字を用いて親しみやすさを出すこと。
- 読者が今日から実践できる具体例が３つ以上紹介されていること。

# 出力形式
- はじめに、記事の良い点を説明します。
- 次に、修正ポイントを箇条書きで出力します。
- これまでの原稿とレビュー結果の履歴がある場合は、その内容も考慮して、直近の原稿をレビューします。

# 注意
- マークダウン表示崩れの防止ルールは自分の出力内容に適用します。レビューの対象項目でありません。

{markdown_rule}
'''

reviewer_agent = LlmAgent(
    name='reviewer_agent',
    model=flash_model,
    description='記事の原稿をレビューするエージェント',
    instruction=instruction,
    include_contents='default',
)

**[AWW-12]**

記事を書き直すかの判断を受け取るタスクノードを定義します。

In [11]:
class HumanCheckResult(BaseModel):
    """記事の書き直しの判断結果"""
    rewrite: bool = Field(description='書き直しの判断結果')

instruction = '''
# タスク
1. レビュー結果に基づいて、記事を書き直すかユーザーに質問します。
2. 得られた情報を HumanCheckResult.rewrite にセットします。
  - 書き直す場合は True
  - 書き直さない場合は False
3. HumanCheckResult.rewrite をセットしたらタスクを終了します。

# 条件
余計な会話はせずに、「レビュー結果に基づいて記事を書き直しますか？」と聞いて、「はい」か「いいえ」の判断を求めてください。
'''

human_check_task = LlmAgent(
    name='human_check_task',
    model=flash_lite_model,
    mode='task',
    output_schema=HumanCheckResult,
    description='記事の書き直しの判断を受け取るタスク',
    instruction=instruction,
)

def process_human_check_result(node_input: HumanCheckResult):
    if node_input.rewrite:
        return Event(
            message='### 記事の書き直しを依頼します。',
            route='rewrite',
        )
    else:
        return Event(
            message='### 記事が完成しました。',
            route='show_last_article',
        )

def show_last_article(ctx, node_input):
    message=f'''
### 完成した記事を表示します。

---

{ctx.state['article_content']}
    '''
    return Event(message=message)

## ワークフローの定義と実行

**[AWW-13]**

ここまでに定義したノードを用いたワークフローを定義します。

In [12]:
article_writing_workflow = Workflow(
    name='article_writing_workflow',
    edges=[
        (
            'START',
            article_information_task, output_article_information,
            research_agent,
            writer_agent, reviewer_agent,
            human_check_task, process_human_check_result,
        ),
        (
            process_human_check_result,
            {
                'rewrite': writer_agent,
                DEFAULT_ROUTE: show_last_article,
            },
        ),
        (
            show_last_article,
        ),
    ],
)

article_writing_app = AdkApp(
    agent=article_writing_workflow,
    app_name='article_writing_app',
)

**[AWW-14]**

ワークフローを実行して、記事を作成します。

In [13]:
chat_client = ChatClient(article_writing_app)

query = '''
秋に向けて、あたらしい趣味を提案する記事を考えています。
「推し活」が流行っているようですが、よりディープな視点のトピックはあるでしょうか？
'''
await chat_client.async_stream_query(query, async_output=True)

/root/.local/lib/python3.13/site-packages/agentplatform/frameworks/adk.py:1146: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/root/.local/lib/python3.13/site-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


**[article_information_task]**

秋に向けて「推し活」をベースにしたディープな新しい趣味の提案、とても魅力的な切り口ですね！

近年、一般的な「アイドルやアニメを応援する・グッズを買う」にとどまらず、体験や探求、自己表現へと進化したディープな推し活が注目されています。秋という季節（芸術の秋、読書の秋、行楽の秋）にも相性が良いトピックとして、以下のような切り口はいかがでしょうか。

---

### ディープな推し活トピックのアイデア

1. **歴史・古典・文豪を推す「知的好奇心の推し活」（読書・芸術の秋）**
   - **内容：** 歴史上の人物（武将、仏像、近代文豪など）を推し、書簡や作品を読み解いたり、ゆかりの寺社や記念館を巡ったりするカルチャー系推し活。
   - **魅力：** 人間関係の消費に疲れた大人が、静かに深く没頭できる秋にぴったりの趣味。

2. **クラフト・ぬい服・ドール作りの「自作・クリエイティブ推し活」（ものづくりの秋）**
   - **内容：** 既製品を買うだけでなく、推しのぬいぐるみ（ぬい）をゼロから自作したり、秋物の衣装を手縫いしたり、推し概念の香水やアクセサリーを調合するハンドメイド。
   - **魅力：** 単なる消費から「自ら手を動かして生み出す趣味」への進化。

3. **保護動物や文化財などを支える「社会貢献型推し活・里親制度」**
   - **内容：** 動物園の動物のサポーター（象主など）になったり、保護猫を月額で見守る活動、地域の伝統工芸や文化財の修復支援など。
   - **魅力：** 「好き」のエネルギーがそのまま誰かの支援につながる、大人の成熟した推し活。

4. **旅と考察を深める「ディープ聖地巡礼・フィールドワーク」（行楽の秋）**
   - **内容：** 映画・アニメ・小説の舞台となった土地へ行き、作中のアングルを探すだけでなく、その土地の歴史・地形・食文化まで徹底的に調査・体験する。

---

### 記事の方向性についてのご相談

記事の骨子を決めるにあたり、以下の点についてご意見をお聞かせいただけますでしょうか？

- **想定するターゲット層（読者）：**
  - （例：一般的な推し活に少し疲れて新しい刺激を求めている20〜30代、じっくり楽しめる大人の趣味を探している30〜50代、など）
- **取り上げたいテーマのイメージ：**
  - 上記のアイデアの中で興味のあるものや、他に組み合わせたいテーマ（「知識系」「手作り系」「おでかけ系」など）はありますでしょうか？

ご希望を伺いながら、記事のテーマと対象読者を固めていければと思います！

In [14]:
query = '''
クリエイティブ系の推し活は面白いですね。ターゲット層の絞り込みが難しそう。
'''
await chat_client.async_stream_query(query, async_output=True)

**[article_information_task]**

クリエイティブ系の推し活、とても熱量が高くて魅力的なジャンルですよね！

ターゲット層を絞り込むにあたって、「どのような動機やライフスタイルの人に届けたいか」によって、いくつかの切り口が考えられます。例えば、以下のようなターゲット設定はいかがでしょうか？

---

### ターゲット層の切り口アイデア

1. **「消費するだけの推し活から、じっくり没頭する趣味にシフトしたい20〜30代女性」**
   - **背景・インサイト：** グッズ購入や現場通いなどのお金・情報スピードの早さに少し疲れ、「推しへの愛をもっと自分のペースで形にしたい」「おうち時間でじっくり集中できる手仕事がしたい」と感じている層。
   - **記事の響き方：** 「秋の夜長に楽しむ、自分だけの推し活クラフト」として癒やしや没頭感を提案できる。

2. **「もともとモノづくりや手芸が好きで、推し活と掛け合わせたい20〜40代」**
   - **背景・インサイト：** 編み物、刺繍、ミニチュア、レジンなどのスキルや興味があり、「既製品にはない自分好みのアイテム（ぬい服や概念アクセ）を作ってみたい」と考えている層。
   - **記事の響き方：** 「大人向けの上質な推し活DIY」「趣味の手芸に新しいテーマを」という切り口で提案できる。

3. **「推し活初心者〜中級者で、秋から手軽に新しいことを始めたい20〜30代」**
   - **背景・インサイト：** 不器用でも簡単にできる「推し概念キャンドル作り」「キットを使ったぬい服作り」など、ハードルの低いクラフトから挑戦してみたい層。
   - **記事の響き方：** 入門編として、初心者でも失敗しないクラフト推し活をステップ形式で提案できる。

---

例えば **「消費する推し活に少し疲れ、秋の夜長に自分のペースで没頭したい20〜30代」** あたりをメインに据えると、読者の共感を呼びやすく、記事のトーンも情緒的で読み応えのあるものにしやすいかと思います。

これらの切り口の中で、イメージに近いものや「もう少しこうしたい」という点はございますでしょうか？

In [15]:
query = '''
案3を採用。
'''
await chat_client.async_stream_query(query, async_output=True)

**[article_information_task]**

案3の「推し活初心者〜中級者で、秋から手軽に新しいことを始めたい20〜30代」ですね！
ハードルが高すぎず、キットや手軽なクラフトを通して「自分で作る楽しさ」を体験できる親しみやすい切り口になりそうです。

こちらを踏まえて、記事の基本情報を以下のようにまとめました。内容をご確認いただけますでしょうか？

---

### 記事の基本情報（案）

- **対象読者（ターゲット層）：**
  推し活初心者〜中級者で、秋の新しい趣味として手軽に始められるものづくりに挑戦したい20〜30代
- **記事のテーマ：**
  秋から始める「クリエイティブ系推し活」入門！初心者でも手軽に楽しめるぬい服・概念クラフトの魅力と始め方

---

こちらの内容でよろしければ確定とさせていただきます。修正や追加のご要望がございましたら、遠慮なくお知らせください！

In [16]:
query = '''
これで決定します。
'''
await chat_client.async_stream_query(query, async_output=True)

**[article_writing_workflow]**


### 記事のテーマとターゲット層を決定しました。
- テーマ: 秋から始める「クリエイティブ系推し活」入門！初心者でも手軽に楽しめるぬい服・概念クラフトの魅力と始め方
- ターゲット層: 推し活初心者〜中級者で、秋の新しい趣味として手軽に始められるものづくりに挑戦したい20〜30代


**[research_agent]**

指定されたテーマおよびターゲット層に向けた記事執筆に役立つ、5つのトピックを選定した調査レポートをまとめました。

---

# 調査レポート：秋から始める「クリエイティブ系推し活」入門

### ■ 記事のターゲット層と狙い
- **ターゲット層：** 推し活初心者〜中級者で、秋のおうち時間や新しい趣味として手軽にものづくりを始めたい **20〜30代**
- **テーマの狙い：** 「手先が器用でないと難しそう」「材料費が高そう」という心理的ハードルを解消し、日常にさりげなく推しを取り入れられる「ぬい服作り」や「概念クラフト」の始め方を分かりやすく提示する。

---

### トピック 1 ： 日常とおしゃれに馴染む「概念クラフト」の魅力とトレンド背景

「概念クラフト」とは、推しの顔写真や公式ロゴを直接使うのではなく、推しのイメージカラー（メンカラ）やモチーフ、楽曲の世界観などを抽象的に落とし込んだハンドメイドアイテムのことです。オフィスや日常生活の中で「オタバレ」を防ぎつつ、さりげなく推しを身近に感じられる点が、仕事とプライベートを両立する **20〜30代** の推し活層から高い支持を集めています。公式グッズ展開が少ない作品やマイナーな推しであっても、自分の解釈を反映して自由にグッズを生み出せることも大きな魅力です。近年はSNSでも自作の概念アイテムをおしゃれなカフェやインテリアとともに撮影し、共有するカルチャーが定着しています。記事では、既製品を買うだけでは味わえない「推しへの解釈を形にするクリエイティブな充足感」を最初のフックとして提示することで、読者の創作意欲を刺激します。

### トピック 2 ： 針と糸を使わない！ 不器用さんでも挫折しない「簡単ぬい服」テクニック

手芸の最大のハードルである「縫う作業」を省き、布用接着剤（コニシの **「裁ほう上手」** など）や布用両面テープを活用したぬい服作りが初心者を中心に人気を博しています。生地を裁断して折り目をアイロンで圧着するだけで、ミシン縫いと同等以上の強度でスカートやパンツ、ポンチョなどの基本服が完成します。さらに、切りっぱなしでも端がほつれにくいフエルトやフリース、ボア素材を選ぶことで、端処理の工程を省略できる点も初心者に最適です。直線裁ちと首元・裾の接着だけで作れる「フード付きケープ」や「マフラー」は、制作時間が **30〜40分** 程度と短く、初めての達成感を得やすいエントリー作品となります。記事内では「工作感覚で楽しめる洋服づくり」として具体的な手順や接着のコツを紹介し、裁縫への苦手意識を解消します。

### トピック 3 ： 100円ショップ（セリア・ダイソー等）で揃う最新素材とコスパ検証

現在、セリアやダイソーなどの大手 **100円ショップ** では「ぬい活」や推し活に特化した専用コーナーが常設されており、手芸店に足を運ばなくても主要な材料が手に入ります。特にセリアでは、ぬい服作りに最適な伸縮性のあるソフトボア生地や、小さなぬいぐるみにぴったりのミニチュア面ファスナー、極小ボタンなどが充実しています。また、ダイソーでも扱いやすいカットクロスやアクセサリー金具、UVレジン液などのクラフト素材が幅広く展開されています。手芸専門店で一から揃えると数千円かかる初期費用も、100均アイテムを中心に活用すれば数百円から **1,000円以内** でスタートできます。記事では「初心者がまず買い揃えるべき100均マストバイ素材リスト」を提示することで、買い出しの失敗を防ぐ実用的な情報を提供します。

### トピック 4 ： 季節感をプラス！ 秋カラー＆ハロウィンテーマの限定アレンジ術

秋はハンドメイドや「ぬい撮り（ぬいぐるみを主役にした写真撮影）」にぴったりのシーズンであり、季節に合わせたテーマ設定が記事の訴求力を高めます。ボルドー、マスタード、テラコッタ、深緑といった深みのあるオータムカラーを取り入れるだけで、いつもの推しカラーがぐっと大人っぽい印象に変化します。また、チェック柄やコーデュロイ、ベロアなどの秋冬素材を用いることで、温かみのある季節限定コーディネートを簡単に表現できます。10月のハロウィンシーズンに向けては、黒や紫の生地を使った「魔女風とんがり帽子」や、白フエルトに目をつけるだけの「おばけケープ」など、季節イベント向けのプチ仮装アイデアが人気です。完成した作品を紅葉スポットや秋のカフェ巡りに連れ出し、秋景色のなかで撮影を楽しむストーリーまで伝えることで、クラフト後の楽しみ方をイメージさせます。

### トピック 5 ： 普段使いできる「概念アクセサリー」の作り方と失敗防止のポイント

概念クラフトの入門として、バッグチャームやキーホルダー、ブレスレットなどの小物・アクセサリー作りはぬい服以上に手軽でおすすめです。レジン液を用いたチャーム作りでは、推しのメンカラで色付けしたレジン液にお好みのラメやホログラム、ドライフラワーを封入するだけで高見えするアイテムが作れます。また、韓国風のビーズキーホルダーやストラップであれば、特別な薬品やライトを使わず、テグスにビーズやアルファベットパーツを通すだけで短時間で完成します。初心者がつまずきやすい「レジンの気泡残り」や「金具の留めミス」などの失敗を防ぐテクニック（爪楊枝を使った気泡潰しや、少量ずつ硬化させる重ね塗りなど）を解説することが重要です。日常の通勤バッグやポーチにさりげなく付けられるデザインの落とし込み方を提案し、20〜30代の読者が胸を張って愛用できる導線を作ります。

**[writer_agent]**

秋の夜長にぴったりな新しい趣味として、推しの世界観を自分の手で表現する「クリエイティブ系推し活」を始めてみませんか？既製品を買うだけでなく、自分の解釈を形にして日常に取り入れる楽しさにハマる人が増えています。

## 推しの世界観をさりげなく楽しむ「概念クラフト」の魅力

最近の推し活トレンドとして注目されているのが、推しの顔写真や公式ロゴを直接使わずに表現する「概念クラフト」です。推しのイメージカラーやモチーフ、楽曲の世界観などを抽象的に落とし込んだアイテムを手作りするスタイルを指します。

- 日常やオフィスで悪目立ちせず、オタバレを防ぎながらさりげなく推しを身近に感じられる
- 公式グッズの展開が少ない作品やマイナーな推しでも、自由にアイテムを生み出せる
- カフェやインテリアにも馴染みやすく、SNSでのおしゃれな発信とも相性が抜群

仕事とプライベートを両立させたい大人世代にとって、自分だけの解釈を形にできるクリエイティブな充足感は格別です。

## 100均素材と「縫わない手芸」でここまで手軽に

ハンドメイドと聞くとハードルが高そうに思えますが、いまや初心者でも挫折しにくい環境が整っています。

セリアやダイソーなどの大手100円ショップには「ぬい活」や推し活に特化した専用コーナーが常設されており、ミニチュアサイズのファスナーや極小ボタン、カットクロスやレジン液などが手に入ります。専門店に行かなくても、1,000円以内の予算で手軽に材料を揃えられるのが嬉しいところです。

さらに、布用接着剤や布用両面テープを活用することで、針や糸を使わずに服や小物を作るテクニックが主流になっています。布を裁断して接着し、アイロンで圧着するだけで、ミシン縫いに匹敵する強度でスカートやケープなどが完成します。

## 30分でできる！秋の「おばけケープ」作りに挑戦

まずは道具も少なく、短時間で達成感が得られる簡単なぬい服作りに挑戦してみましょう。端処理が不要でほつれにくいフエルトを使えば、工作感覚でハロウィン風のケープが作れます。

- 白や黒のフエルトを直線裁ちし、首元にリボンを通すか接着剤で留めるパーツを作る
- 首元と裾を布用接着剤で整え、30分から40分ほどでミニケープを仕上げる
- ハロウィンを意識しておばけの目をフエルトで貼り付けたり、秋らしいチェック柄のリボンを添える

針も糸も使わずにあっという間に完成するので、不器用さんでも安心です。出来上がったケープをぬいぐるみに着せて、紅葉やカフェ巡りの「ぬい撮り」に出かけてみてください。

## 自分だけの「推し表現」で秋を楽しもう

クリエイティブ系推し活の魅力は、誰かの正解を追うのではなく、自分だけの解釈や愛情を自由に形にできる点にあります。プチプラ素材と便利な道具を使えば、お金や時間をかけすぎずに心地よいものづくり時間が過ごせます。この秋は、日常にさりげなく寄り添うオリジナルの推しアイテムを手作りしてみてはいかがでしょうか。

**[reviewer_agent]**

ご提示いただいた原稿について、記事の条件に照らし合わせてレビューいたしました。

---

### ■ 記事の良い点

- **ターゲットに刺さる魅力的なテーマ設定：**
  「グッズを買う」だけでなく「解釈を形にする」という、一歩進んだディープな推し活の面白さがしっかりと伝わってきます。オタバレ防止や日常使いといった大人世代のニーズを捉えた視点が秀逸です。
- **心理的ハードルを下げる工夫：**
  「 **100円ショップ** で揃う」「針と糸を使わない」といった手軽さを提示することで、手芸未経験の読者でも「自分にもできそう」と思わせる工夫がなされています。
- **季節感を取り入れた提案：**
  秋の夜長やおうち時間、ハロウィンを意識した「おばけケープ」など、季節ならではの楽しみ方が盛り込まれており、読後に行動したくなる魅力があります。

---

### ■ 修正ポイント

記事の条件を満たし、より読者を引き込むコンテンツにするために、以下の点を修正してください。

1. **マークダウンヘッダ（ `#` ）付きのタイトルを冒頭に新設する：**
   現在、原稿に大見出し（タイトル）がありません。読者が「すぐに読まなきゃ」と感じる、今日から役立つ生活情報を含んだ **40文字程度** のタイトルを `#` を付けて一番上に追加してください。
   - *（例案）* `# 今日の帰りに100均へ！秋の夜長を彩る「縫わない推し活クラフト」超入門` （37文字）

2. **「なぜいまこのテーマなのか」を伝える導入の強化：**
   タイトルの直後に、なぜ秋の今このクラフトを始めるべきなのか（おうち時間が増える秋の気候、100均各社で推し活コーナーが最も充実している今だからこそ、など）を明記し、読者が「今すぐ読むべき理由」を強く動機づけてください。

3. **各セクションのサブタイトル（ `##` ）に絵文字を追加する：**
   親しみやすさを出し、視認性を高めるために、すべての小見出しに内容に合った絵文字を取り入れてください。
   - *（例）* `## 🎨 推しの世界観をさりげなく楽しむ「概念クラフト」の魅力`
   - *（例）* `## ✂️ 100均素材と「縫わない手芸」でここまで手軽に`

4. **今日から実践できる具体例を「3つ以上」に増やす：**
   現在の原稿では「おばけケープ」の1点のみが具体例となっています。調査レポートにあるトピックを活かし、今日から挑戦できる具体的なクラフトアイデアを **3つ以上** 明記してください。
   - *（追加アイデア例）*
     1. **針・糸なしで作る「おばけケープ」** （フェルトと布用接着剤を活用）
     2. **100均パーツを通すだけ！「メンカラのビーズチャーム」** （手軽に作れて通勤バッグに付けられる）
     3. **秋色を閉じ込める「概念レジンキーホルダー」** （ドライフラワーやラメを使った高見えアクセ）

**[human_check_task]**

レビュー結果に基づいて記事を書き直しますか？

In [17]:
query = '''
書き直してください。
'''
await chat_client.async_stream_query(query, async_output=True)

**[article_writing_workflow]**

### 記事の書き直しを依頼します。

**[writer_agent]**

# 今日の帰りに100均へ！秋の夜長を彩る「縫わない推し活クラフト」超入門

涼しくなっておうち時間が増える秋は、100円ショップ各社で推し活グッズが最も充実する今こそ新しい趣味を始める絶好のチャンスです。日常に推しをさりげなく溶け込ませる「クリエイティブ系推し活」の扉を、今日の帰り道から開いてみませんか？

## 🎨 推しの世界観をさりげなく楽しむ「概念クラフト」の魅力

最近の推し活トレンドとして注目されているのが、推しの顔写真や公式ロゴを直接使わずに表現する「概念クラフト」です。推しのイメージカラーやモチーフ、楽曲の世界観などを抽象的に落とし込んだアイテムを手作りするスタイルを指します。

- オフィスや日常生活でオタバレを防ぎつつ、さりげなく推しを身近に感じられる
- 公式グッズの展開が少ない作品やマイナーな推しでも、自由にアイテムを生み出せる
- カフェやインテリアにも馴染みやすく、SNSでおしゃれに発信できる

仕事とプライベートを両立させたい大人世代にとって、既製品を買うだけでなく「推しへの解釈を形にする充足感」は格別です。

## ✂️ 100均進化と「縫わない手芸」でここまで手軽に

ハンドメイドと聞くとハードルが高そうに思えますが、いまや初心者でも挫折しにくい環境が整っています。

セリアやダイソーなどの大手100円ショップには推し活やぬい活に特化した専用コーナーが常設されており、伸縮性のあるボア生地や極小ボタン、UVレジン液などが手に入ります。手芸店で一から揃えると数千円かかる材料費も、100均を活用すれば1,000円以内からスタート可能です。

さらに、布用接着剤や布用両面テープを活用することで、針や糸を使わずに服や小物を作るテクニックが定着しています。布を裁断して接着し、アイロンで圧着するだけで、ミシン縫いと同等以上の強度でしっかり仕上がります。

## 🧵 今日からすぐ作れる！初心者向けクラフトアイデア3選

手先の器用さに自信がなくても、短時間で達成感が得られる具体的なおすすめクラフトを3つ紹介します。

- 針・糸なしで作る「おばけケープ」
端がほつれないフエルト生地を直線裁ちし、布用接着剤で首元や裾を留めるだけで作れるぬい服です。白や黒のフエルトに目をつければ、制作時間30分から40分ほどでハロウィン向けのプチ仮装が完成します。

- 100均パーツを通すだけ！「メンカラのビーズチャーム」
特別な工具や薬品を使わず、テグスに推しカラーのビーズやアルファベットパーツを通すだけで作れる韓国風チャームです。通勤バッグやポーチにさりげなく付けられるデザインに仕上げやすく、普段使いに重宝します。

- 秋色を閉じ込める「概念レジンキーホルダー」
推しカラーに着色したレジン液に、ボルドーやマスタードなどの秋色ラメ、ドライフラワーを封入して固める高見えアクセサリーです。爪楊枝で丁寧に気泡を潰し、少量ずつ重ね塗りして硬化させることで、失敗せず綺麗に仕上がります。

## 🍂 自分だけの表現で秋の推し活をもっと豊かに

クリエイティブ系推し活の魅力は、誰かの正解を追うのではなく、自分だけの解釈や愛情を自由に形にできる点にあります。プチプラ素材と便利な道具を使えば、お金や時間をかけすぎずに心地よいものづくり時間を満喫できます。完成した作品を連れて秋のカフェや紅葉スポットへお出かけし、自分だけの特別な秋を楽しんでみてください。

**[reviewer_agent]**

再提出いただいた原稿をレビューいたしました。前回の修正ポイントが的確に反映され、記事の魅力と実用性が大幅に向上しています。

---

### ■ 記事の良い点

- **引きの強いタイトルと明確な導入：**
  「今日の帰りに100均へ！」という具体的な行動喚起と、37文字という適切な文字数で読者の関心を掴めています。また、タイトルの直後で「なぜ秋の今なのか（おうち時間の増加、100均の推し活グッズ充実）」が明確に提示されており、読み進める動機づけが非常にスムーズです。
- **視認性と親しみやすさの向上：**
  すべてのサブタイトル（ `##` ）に内容に即した絵文字が配置され、スマホなどの画面でもメリハリがついて読みやすくなりました。
- **実践しやすい3つの具体例：**
  「おばけケープ」「ビーズチャーム」「レジンキーホルダー」と、難易度や用途（ぬい服・バッグ・アクセ）の異なる具体例が **3つ** 提示されており、読者が自分に合ったものを選びやすい構成になっています。

---

### ■ 修正ポイント（さらなるブラッシュアップに向けて）

記事の条件はすべてクリアできていますが、Web読み物としての可読性と実用性をさらに高めるための改善点を挙げます。

1. **具体例3選のレイアウト・視認性の改善：**
   「今日からすぐ作れる！初心者向けクラフトアイデア3選」のセクションについて、箇条書きのタイトル部分を太字にする、または改行を整えることで、パッと見の読みやすさを高めましょう。
   - *（改善案）*
     - **① 針・糸なしで作る「おばけケープ」**
       端がほつれないフエルト生地を直線裁ちし、布用接着剤で首元や裾を留めるだけで作れるぬい服です。白や黒のフエルトに目をつければ、制作時間 **30〜40分** ほどでハロウィン向けのプチ仮装が完成します。
     - **② 100均パーツを通すだけ！「メンカラのビーズチャーム」**
       特別な工具や薬品を使わず、テグスに推しカラーのビーズやアルファベットパーツを通すだけで作れる韓国風チャームです。通勤バッグやポーチにさりげなく付けられるデザインに仕上げやすく、普段使いに重宝します。
     - **③ 秋色を閉じ込める「概念レジンキーホルダー」**
       推しカラーに着色したレジン液に、ボルドーやマスタードなどの秋色ラメ、ドライフラワーを封入して固める高見えアクセサリーです。爪楊枝で丁寧に気泡を潰し、少量ずつ重ね塗りして硬化させることで、失敗せず綺麗に仕上がります。

2. **「100均のどこを探せばいいか」のワンアクションを追加：**
   「今日の帰りに100均へ！」という導入に合わせて、「手芸コーナーや推し活コーナー、ハロウィン特設売り場をチェックしてみて」といった、店舗で迷わないための一言を添えると、読者が実際に行動に移しやすくなります。

**[human_check_task]**

レビュー結果に基づいて記事を書き直しますか？

In [18]:
query = '''
もう一度、書き直してください。
'''
await chat_client.async_stream_query(query, async_output=True)

**[article_writing_workflow]**

### 記事の書き直しを依頼します。

**[writer_agent]**

# 今日の帰りに100均へ！秋の夜長を彩る「縫わない推し活クラフト」超入門

涼しくなっておうち時間が増える秋は、100円ショップ各社で推し活グッズが最も充実する今こそ新しい趣味を始める絶好のチャンスです。日常に推しをさりげなく溶け込ませる「クリエイティブ系推し活」の扉を、今日の帰り道から開いてみませんか？

## 🎨 推しの世界観をさりげなく楽しむ「概念クラフト」の魅力

最近の推し活トレンドとして注目されているのが、推しの顔写真や公式ロゴを直接使わずに表現する「概念クラフト」です。推しのイメージカラーやモチーフ、楽曲の世界観などを抽象的に落とし込んだアイテムを手作りするスタイルを指します。

- オフィスや日常生活でオタバレを防ぎつつ、さりげなく推しを身近に感じられる
- 公式グッズの展開が少ない作品やマイナーな推しでも、自由にアイテムを生み出せる
- カフェやインテリアにも馴染みやすく、SNSでおしゃれに発信できる

仕事とプライベートを両立させたい大人世代にとって、既製品を買うだけでなく「推しへの解釈を形にする充足感」は格別です。

## ✂️ 100均進化と「縫わない手芸」でここまで手軽に

ハンドメイドと聞くとハードルが高そうに思えますが、いまや初心者でも挫折しにくい環境が整っています。

セリアやダイソーなどの大手100円ショップに入ったら、まずは手芸コーナーや推し活コーナー、そして秋のハロウィン特設売り場をチェックしてみてください。伸縮性のあるボア生地や極小ボタン、UVレジン液などが驚くほど豊富に並んでいます。手芸店で一から揃えると数千円かかる材料費も、100均を活用すれば **1,000円以内** からスタート可能です。

さらに、布用接着剤や布用両面テープを活用することで、針や糸を使わずに服や小物を作るテクニックが定着しています。布を裁断して接着し、アイロンで圧着するだけで、ミシン縫いと同等以上の強度でしっかり仕上がります。

## 🧵 今日からすぐ作れる！初心者向けクラフトアイデア3選

手先の器用さに自信がなくても、短時間で達成感が得られる具体的なおすすめクラフトを3つ紹介します。

- **① 針・糸なしで作るおばけケープ**
  端がほつれないフエルト生地を直線裁ちし、布用接着剤で首元や裾を留めるだけで作れるぬい服です。白や黒のフエルトに目をつければ、制作時間 **30〜40分** ほどでハロウィン向けのプチ仮装が完成します。

- **② 100均パーツを通すだけのメンカラビーズチャーム**
  特別な工具や薬品を使わず、テグスに推しカラーのビーズやアルファベットパーツを通すだけで作れる韓国風チャームです。通勤バッグやポーチにさりげなく付けられるデザインに仕上げやすく、普段使いに重宝します。

- **③ 秋色を閉じ込める概念レジンキーホルダー**
  推しカラーに着色したレジン液に、ボルドーやマスタードなどの秋色ラメ、ドライフラワーを封入して固める高見えアクセサリーです。爪楊枝で丁寧に気泡を潰し、少量ずつ重ね塗りして硬化させることで、失敗せず綺麗に仕上がります。

## 🍂 自分だけの表現で秋の推し活をもっと豊かに

クリエイティブ系推し活の魅力は、誰かの正解を追うのではなく、自分だけの解釈や愛情を自由に形にできる点にあります。プチプラ素材と便利な道具を使えば、お金や時間をかけすぎずに心地よいものづくり時間を満喫できます。完成した作品を連れて秋のカフェや紅葉スポットへお出かけし、自分だけの特別な秋を楽しんでみてください。

**[reviewer_agent]**

再提出いただいた原稿をレビューいたしました。

前回のレビューで提案した「売り場への具体的な誘導」や「具体例の視認性向上」が見事に反映され、非常に完成度の高い記事に仕上がっています。

---

### ■ 記事の良い点

- **すべての必須条件を高い水準で達成：**
  - **タイトル：** 「今日の帰りに100均へ！」という即効性のあるコピーと **37文字** の適切な長さで、マークダウンヘッダ（ `#` ）も正しく設定されています。
  - **導入：** 秋という季節性や、100均の品揃えが充実するタイミングを捉えて「いま読むべき理由」が明確に伝わります。
  - **サブタイトル：** 各見出しにテーマに合った絵文字が配され、流し読みでも内容が直感的に把握できます。
  - **具体例：** 初心者向けの実践アイデアが **3つ** 、太字とインデントを用いて分かりやすく整理されています。
- **読者の行動を後押しする細やかな配慮：**
  「手芸コーナーや推し活コーナー、そして秋のハロウィン特設売り場をチェック」という具体的な導線が加わったことで、読者が店舗に入ってから迷わず材料を探せる実用的な記事になっています。

---

### ■ 修正ポイント（公開・配信に向けた最終確認）

基本要件はすべてクリアしており、このまま公開できるクオリティに達しています。さらに読者のエンゲージメントを高めるための最終調整として、以下の軽微なポイントをご検討ください。

- **「所要時間」や「難易度」のアイコン化・追記（任意）：**
  具体例の3選について、①の「おばけケープ」には「制作時間 **30〜40分** 」と記載されています。②のビーズチャームや③のレジンキーホルダーにも、「制作時間：約 **15分** 」「難易度：★☆☆」のような目安時間を追記すると、忙しい読者が今日どれを試すか直感的に選びやすくなります。
- **SNSシェアや読者アクションを促す結びの一言（任意）：**
  最終段落の末尾に「作ったアイテムはハッシュタグを付けて投稿してみては？」といった読者同士の交流を促す一文を軽く添えると、SNSでの拡散や記事の定着に寄与します。

**[human_check_task]**

レビュー結果に基づいて記事を書き直しますか？

In [19]:
query = '''
これで完成です。
'''
await chat_client.async_stream_query(query, async_output=True)

**[article_writing_workflow]**

### 記事が完成しました。

**[article_writing_workflow]**


### 完成した記事を表示します。

---

# 今日の帰りに100均へ！秋の夜長を彩る「縫わない推し活クラフト」超入門

涼しくなっておうち時間が増える秋は、100円ショップ各社で推し活グッズが最も充実する今こそ新しい趣味を始める絶好のチャンスです。日常に推しをさりげなく溶け込ませる「クリエイティブ系推し活」の扉を、今日の帰り道から開いてみませんか？

## 🎨 推しの世界観をさりげなく楽しむ「概念クラフト」の魅力

最近の推し活トレンドとして注目されているのが、推しの顔写真や公式ロゴを直接使わずに表現する「概念クラフト」です。推しのイメージカラーやモチーフ、楽曲の世界観などを抽象的に落とし込んだアイテムを手作りするスタイルを指します。

- オフィスや日常生活でオタバレを防ぎつつ、さりげなく推しを身近に感じられる
- 公式グッズの展開が少ない作品やマイナーな推しでも、自由にアイテムを生み出せる
- カフェやインテリアにも馴染みやすく、SNSでおしゃれに発信できる

仕事とプライベートを両立させたい大人世代にとって、既製品を買うだけでなく「推しへの解釈を形にする充足感」は格別です。

## ✂️ 100均進化と「縫わない手芸」でここまで手軽に

ハンドメイドと聞くとハードルが高そうに思えますが、いまや初心者でも挫折しにくい環境が整っています。

セリアやダイソーなどの大手100円ショップに入ったら、まずは手芸コーナーや推し活コーナー、そして秋のハロウィン特設売り場をチェックしてみてください。伸縮性のあるボア生地や極小ボタン、UVレジン液などが驚くほど豊富に並んでいます。手芸店で一から揃えると数千円かかる材料費も、100均を活用すれば **1,000円以内** からスタート可能です。

さらに、布用接着剤や布用両面テープを活用することで、針や糸を使わずに服や小物を作るテクニックが定着しています。布を裁断して接着し、アイロンで圧着するだけで、ミシン縫いと同等以上の強度でしっかり仕上がります。

## 🧵 今日からすぐ作れる！初心者向けクラフトアイデア3選

手先の器用さに自信がなくても、短時間で達成感が得られる具体的なおすすめクラフトを3つ紹介します。

- **① 針・糸なしで作るおばけケープ**
  端がほつれないフエルト生地を直線裁ちし、布用接着剤で首元や裾を留めるだけで作れるぬい服です。白や黒のフエルトに目をつければ、制作時間 **30〜40分** ほどでハロウィン向けのプチ仮装が完成します。

- **② 100均パーツを通すだけのメンカラビーズチャーム**
  特別な工具や薬品を使わず、テグスに推しカラーのビーズやアルファベットパーツを通すだけで作れる韓国風チャームです。通勤バッグやポーチにさりげなく付けられるデザインに仕上げやすく、普段使いに重宝します。

- **③ 秋色を閉じ込める概念レジンキーホルダー**
  推しカラーに着色したレジン液に、ボルドーやマスタードなどの秋色ラメ、ドライフラワーを封入して固める高見えアクセサリーです。爪楊枝で丁寧に気泡を潰し、少量ずつ重ね塗りして硬化させることで、失敗せず綺麗に仕上がります。

## 🍂 自分だけの表現で秋の推し活をもっと豊かに

クリエイティブ系推し活の魅力は、誰かの正解を追うのではなく、自分だけの解釈や愛情を自由に形にできる点にあります。プチプラ素材と便利な道具を使えば、お金や時間をかけすぎずに心地よいものづくり時間を満喫できます。完成した作品を連れて秋のカフェや紅葉スポットへお出かけし、自分だけの特別な秋を楽しんでみてください。
    

**[AWW-15]**

セッションのステートに格納された完成原稿のマークダウンテキストを取得します。

In [21]:
session = await chat_client.adk_app.async_get_session(
    user_id = chat_client.user_id,
    session_id = chat_client.session_id,
)

print(session.state['article_content'])

# 今日の帰りに100均へ！秋の夜長を彩る「縫わない推し活クラフト」超入門

涼しくなっておうち時間が増える秋は、100円ショップ各社で推し活グッズが最も充実する今こそ新しい趣味を始める絶好のチャンスです。日常に推しをさりげなく溶け込ませる「クリエイティブ系推し活」の扉を、今日の帰り道から開いてみませんか？

## 🎨 推しの世界観をさりげなく楽しむ「概念クラフト」の魅力

最近の推し活トレンドとして注目されているのが、推しの顔写真や公式ロゴを直接使わずに表現する「概念クラフト」です。推しのイメージカラーやモチーフ、楽曲の世界観などを抽象的に落とし込んだアイテムを手作りするスタイルを指します。

- オフィスや日常生活でオタバレを防ぎつつ、さりげなく推しを身近に感じられる
- 公式グッズの展開が少ない作品やマイナーな推しでも、自由にアイテムを生み出せる
- カフェやインテリアにも馴染みやすく、SNSでおしゃれに発信できる

仕事とプライベートを両立させたい大人世代にとって、既製品を買うだけでなく「推しへの解釈を形にする充足感」は格別です。

## ✂️ 100均進化と「縫わない手芸」でここまで手軽に

ハンドメイドと聞くとハードルが高そうに思えますが、いまや初心者でも挫折しにくい環境が整っています。

セリアやダイソーなどの大手100円ショップに入ったら、まずは手芸コーナーや推し活コーナー、そして秋のハロウィン特設売り場をチェックしてみてください。伸縮性のあるボア生地や極小ボタン、UVレジン液などが驚くほど豊富に並んでいます。手芸店で一から揃えると数千円かかる材料費も、100均を活用すれば **1,000円以内** からスタート可能です。

さらに、布用接着剤や布用両面テープを活用することで、針や糸を使わずに服や小物を作るテクニックが定着しています。布を裁断して接着し、アイロンで圧着するだけで、ミシン縫いと同等以上の強度でしっかり仕上がります。

## 🧵 今日からすぐ作れる！初心者向けクラフトアイデア3選

手先の器用さに自信がなくても、短時間で達成感が得られる具体的なおすすめクラフトを3つ紹介します。

- **① 針・糸なしで作るおばけケープ**
  端がほつれないフエルト生地を直線裁ちし、布用接着剤で首元や裾を留めるだけで作れるぬい服です。白や黒のフエルトに